# Project 14: Airline Flight Delay & On-Time Performance
**Industry:** Aviation / Transportation  
**Goal:** Study flight schedules, departure delays and route performance to find bottleneck airports and delay patterns.  

All code is in this single notebook (tables and printed results only, no charts).  
Run **Kernel → Restart & Run All**. If `14_flight_delays.csv` is not found, the notebook creates a 1000-row sample file automatically.

In [ ]:
import os
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# ---- Project 14 column names (edit here if your CSV differs) ----
C14 = dict(id="Flight_ID", date="Flight_Date", airline="Airline", origin="Origin", dest="Destination",
           dep_time="Scheduled_Departure_Time", dep_delay="Departure_Delay_Mins",
           arr_delay="Arrival_Delay_Mins", cancelled="Cancelled", diverted="Diverted", cause="Delay_Cause")

CSV_NAME = "14_flight_delays.csv"

def find_csv(name):
    """Look for the CSV in the current folder, ./data, ../data, or anywhere below the current folder."""
    import glob
    for pattern in [name, f"data/{name}", f"../data/{name}", f"**/{name}"]:
        hits = glob.glob(pattern, recursive=True)
        if hits:
            return hits[0]
    return None

def make_sample_flights(n=1000, seed=42):
    """Creates a reproducible sample flight dataset (with duplicates and missing values to clean)."""
    rng = np.random.default_rng(seed)
    airports = ["DEL", "BOM", "BLR", "MAA", "HYD", "CCU", "AMD", "PNQ", "GOI", "COK"]
    bias = dict(zip(airports, [14, 12, 6, 4, 2, 8, 0, -1, 3, -2]))
    airlines = ["IndiGo", "Air India", "SpiceJet", "Vistara", "Akasa Air", "GoAir"]
    causes = ["Weather", "Carrier", "Air Traffic", "Security", "Late Aircraft"]
    f = pd.DataFrame({"Flight_ID": [f"FL{100000 + i}" for i in range(n)]})
    f["Flight_Date"] = pd.to_datetime("2026-01-01") + pd.to_timedelta(rng.integers(0, 330, n), unit="D")
    f["Airline"] = rng.choice(airlines, n, p=[.35, .2, .15, .12, .1, .08])
    f["Origin"] = rng.choice(airports, n, p=[.2, .17, .14, .1, .09, .08, .06, .06, .05, .05])
    f["Destination"] = [rng.choice([a for a in airports if a != o]) for o in f["Origin"]]
    w = np.array([5, 8, 9, 8, 7, 6, 6, 6, 7, 8, 9, 10, 9, 8, 7, 5, 4, 3, 2]); w = w / w.sum()
    hours = rng.choice(range(5, 24), n, p=w)
    f["Scheduled_Departure_Time"] = [f"{h:02d}:{rng.choice([0, 15, 30, 45]):02d}" for h in hours]
    tod = np.where(hours >= 17, 8, np.where(hours >= 12, 4, 0)) + np.where(hours >= 21, 3, 0)
    dep = rng.gamma(1.6, 8, n) + f["Origin"].map(bias).values + tod - 8
    dep = np.round(np.clip(dep, -10, None), 0)
    spikes = rng.random(n) < 0.04
    dep[spikes] += rng.integers(90, 300, spikes.sum())
    f["Departure_Delay_Mins"] = dep
    f["Arrival_Delay_Mins"] = np.round(dep * 0.9 + rng.normal(0, 9, n) - 2, 0)
    f["Distance_KM"] = rng.integers(300, 2500, n)
    f["Cancelled"] = np.where(rng.random(n) < 0.025, "Yes", "No")
    f["Diverted"] = np.where(rng.random(n) < 0.01, "Yes", "No")
    f["Delay_Cause"] = np.where(dep > 15, rng.choice(causes, n, p=[.22, .28, .25, .05, .20]), "No Delay")
    f.loc[rng.choice(n, 45, replace=False), "Arrival_Delay_Mins"] = np.nan          # missing values
    f = pd.concat([f.iloc[:n - 15], f.sample(15, random_state=1)], ignore_index=True)  # duplicate IDs
    f["Flight_Date"] = f["Flight_Date"].dt.strftime("%Y-%m-%d")
    return f

def to_bool(s):
    """Convert 0/1, True/False, Yes/No to boolean."""
    return s.astype(str).str.strip().str.lower().isin(["1", "1.0", "true", "yes", "y"])

## Task 1: Load the CSV and inspect columns, shapes and data types

In [ ]:
csv_path = find_csv(CSV_NAME)
if csv_path is None:
    os.makedirs("data", exist_ok=True)
    csv_path = os.path.join("data", CSV_NAME)
    make_sample_flights().to_csv(csv_path, index=False)
    print(f"{CSV_NAME} not found -> created a 1000-row sample file at {csv_path}")
else:
    print("Loaded:", csv_path)

df = pd.read_csv(csv_path)
c = C14
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
df.head()

## Task 2: Missing values and duplicate Flight_ID entries

In [ ]:
print("Missing values per column:")
print(df.isna().sum())

dup_ids = df.duplicated(subset=c["id"]).sum()
print(f"\nDuplicate {c['id']} entries: {dup_ids}")

df = df.drop_duplicates(subset=c["id"], keep="first").reset_index(drop=True)
print("Shape after removing duplicates (kept first):", df.shape)

## Task 3: Handle missing Arrival_Delay_Mins
**Method:** fill with the **median arrival delay of the same route** (fallback: overall median).  
**Justification:** delays are right-skewed with extreme values, so the median is robust (the mean would be inflated), and using the route keeps local context.

In [ ]:
df["Route"] = df[c["origin"]].astype(str) + " -> " + df[c["dest"]].astype(str)

missing_before = df[c["arr_delay"]].isna().sum()
route_median = df.groupby("Route")[c["arr_delay"]].transform("median")
df[c["arr_delay"]] = df[c["arr_delay"]].fillna(route_median)
df[c["arr_delay"]] = df[c["arr_delay"]].fillna(df[c["arr_delay"]].median())

print(f"Missing before: {missing_before} | after: {df[c['arr_delay']].isna().sum()}")

## Task 4: Convert Flight_Date to datetime; extract month and day of week

In [ ]:
df[c["date"]] = pd.to_datetime(df[c["date"]], errors="coerce")
df["Month"] = df[c["date"]].dt.month
df["Month_Name"] = df[c["date"]].dt.month_name()
df["Day_Of_Week"] = df[c["date"]].dt.day_name()
df[[c["date"], "Month", "Month_Name", "Day_Of_Week"]].head()

## Task 5: Total flights, average departure delay, on-time % (delay ≤ 15 mins)

In [ ]:
total_flights = len(df)
avg_dep_delay = df[c["dep_delay"]].mean()
on_time_pct = (df[c["arr_delay"]] <= 15).mean() * 100

print(f"Total flights:           {total_flights}")
print(f"Average departure delay: {avg_dep_delay:.2f} mins")
print(f"On-time % (<= 15 mins):  {on_time_pct:.2f}%")

## Task 6: Rank origin airports by average departure delay

In [ ]:
airport_rank = (df.groupby(c["origin"])[c["dep_delay"]]
                  .agg(avg_dep_delay="mean", flights="count")
                  .sort_values("avg_dep_delay", ascending=False).round(2))
airport_rank.insert(0, "rank", range(1, len(airport_rank) + 1))
airport_rank

## Task 7: Top 10 routes (Origin → Destination) by average arrival delay

In [ ]:
route_rank = (df.groupby("Route")[c["arr_delay"]]
                .agg(avg_arr_delay="mean", flights="count")
                .sort_values("avg_arr_delay", ascending=False).round(2))
route_rank.head(10)

## Task 8: Delay rates across times of day (Morning, Afternoon, Evening, Night)

In [ ]:
def get_hour(series):
    s = series.astype(str).str.strip()
    t = pd.to_datetime(s, format="%H:%M", errors="coerce")       # "HH:MM"
    hour = t.dt.hour
    if hour.notna().mean() < 0.5:                                # try numbers like 1435
        hour = pd.to_numeric(series, errors="coerce") // 100
    return hour

def time_bucket(h):
    if pd.isna(h):  return np.nan
    if 5 <= h < 12:  return "Morning"      # 05:00-11:59
    if 12 <= h < 17: return "Afternoon"    # 12:00-16:59
    if 17 <= h < 21: return "Evening"      # 17:00-20:59
    return "Night"                         # 21:00-04:59

df["Hour"] = get_hour(df[c["dep_time"]])
df["Time_Of_Day"] = df["Hour"].apply(time_bucket)

tod = df.groupby("Time_Of_Day").agg(
    flights=(c["id"], "count"),
    avg_dep_delay=(c["dep_delay"], "mean"),
    avg_arr_delay=(c["arr_delay"], "mean"),
    delay_rate_pct=(c["arr_delay"], lambda x: (x > 15).mean() * 100),
).round(2).reindex(["Morning", "Afternoon", "Evening", "Night"])
tod

## Task 9: Breakdown of delay causes

In [ ]:
causes = df.loc[df[c["cause"]] != "No Delay", c["cause"]].value_counts()
pd.DataFrame({"count": causes, "pct": (causes / causes.sum() * 100).round(2)})

## Task 10: Cancellation and diversion percentages by airline

In [ ]:
df["_cancelled"] = to_bool(df[c["cancelled"]])
df["_diverted"] = to_bool(df[c["diverted"]])

airline_stats = (df.groupby(c["airline"]).agg(
    flights=(c["id"], "count"),
    cancel_pct=("_cancelled", lambda x: x.mean() * 100),
    divert_pct=("_diverted", lambda x: x.mean() * 100),
).round(2).sort_values("cancel_pct", ascending=False))
airline_stats

## Task 11: Correlation between Departure_Delay_Mins and Arrival_Delay_Mins

In [ ]:
corr = df[c["dep_delay"]].corr(df[c["arr_delay"]])
print(f"Pearson correlation: {corr:.3f}")

## Task 12: NumPy median, standard deviation and 90th percentile of departure delays

In [ ]:
dep = df[c["dep_delay"]].dropna().to_numpy()
print(f"Median:             {np.median(dep):.2f} mins")
print(f"Standard deviation: {np.std(dep, ddof=1):.2f} mins")
print(f"90th percentile:    {np.percentile(dep, 90):.2f} mins")

## Task 13: Extreme delay outliers using the IQR method

In [ ]:
q1, q3 = np.percentile(dep, [25, 75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

outliers = df[(df[c["dep_delay"]] < lower) | (df[c["dep_delay"]] > upper)]
print(f"Q1 = {q1:.1f}, Q3 = {q3:.1f}, IQR = {iqr:.1f}")
print(f"Fences: [{lower:.1f}, {upper:.1f}]")
print(f"Outliers: {len(outliers)} ({len(outliers) / len(df) * 100:.2f}% of flights)")
outliers.nlargest(10, c["dep_delay"])[[c["id"], "Route", c["dep_delay"], c["arr_delay"]]]

## Task 14: Delay-severity category column
**Documented thresholds (arrival delay, minutes):** On-Time ≤ 15 | Minor 16–30 | Moderate 31–60 | Major 61–120 | Severe > 120

In [ ]:
bins = [-np.inf, 15, 30, 60, 120, np.inf]
labels = ["On-Time", "Minor", "Moderate", "Major", "Severe"]
df["Delay_Severity"] = pd.cut(df[c["arr_delay"]], bins=bins, labels=labels)

sev = df["Delay_Severity"].value_counts().reindex(labels)
pd.DataFrame({"flights": sev, "pct": (sev / sev.sum() * 100).round(2)})

## Task 15: Five operational recommendations (built from the results above)

In [ ]:
worst_airport = airport_rank.index[0]
worst_route = route_rank.index[0]
worst_tod = tod["avg_dep_delay"].idxmax()
best_tod = tod["avg_dep_delay"].idxmin()
top_cause = causes.index[0]

recs = [
    f"1. Target bottleneck airport {worst_airport} (avg departure delay {airport_rank.iloc[0]['avg_dep_delay']} mins): "
    "add ground-crew capacity and speed up gate turnaround.",
    f"2. Re-time or add buffer to the worst route {worst_route} (avg arrival delay {route_rank.iloc[0]['avg_arr_delay']} mins) "
    "so schedules are realistic.",
    f"3. Space out departures in the {worst_tod} slot (highest delays) and use the {best_tod} slot for critical connections.",
    f"4. Departure and arrival delays are strongly correlated (r = {corr:.2f}); fix problems at departure "
    "(boarding, baggage, crew readiness) so delays do not propagate downstream.",
    f"5. The leading delay cause is {top_cause} ({causes.iloc[0]} flights): create a dedicated action plan for it and review "
    f"extreme outliers (> {upper:.0f} mins) weekly.",
]
for r in recs:
    print(r, "\n")

### Save cleaned data and key tables (optional)

In [ ]:
os.makedirs("output", exist_ok=True)
df.to_csv("output/flight_delays_cleaned.csv", index=False)
airport_rank.to_csv("output/airport_delay_ranking.csv")
route_rank.head(10).to_csv("output/worst_routes.csv")
tod.to_csv("output/time_of_day_delays.csv")
print("Saved files in the output folder.")